# Notebook 04: Positional Encodings

## Why This Matters

Attention is permutation-invariant: if you shuffle the tokens in a sequence, the self-attention operation produces the same output (just shuffled). This means a transformer has no inherent sense of order — "the cat sat on the mat" and "mat the on sat cat the" would produce identical attention patterns without positional information. Positional encodings are the solution, and the specific choice of positional encoding has a massive impact on a model's ability to generalize to longer sequences than seen during training. RoPE (Rotary Position Embeddings), which powers LLaMA, Mistral, and Gemma, is now the dominant approach — and understanding why requires working through the math of why sinusoidal and learned embeddings fail at extrapolation.

## Background

The positional encoding problem has been solved in progressively better ways since 2017:

1. **Sinusoidal PE** (Vaswani 2017): deterministic, allows some extrapolation but attention patterns degrade at unseen lengths
2. **Learned PE** (GPT, BERT): simple embedding lookup, works well but hard cutoff at max training length  
3. **Relative PE** (Shaw et al. 2018, T5): encode distance between positions rather than absolute positions
4. **RoPE** (Su et al. 2021): rotate Q, K vectors; relative positions emerge naturally in dot products
5. **ALiBi** (Press et al. 2021): add position-dependent bias to attention scores; strong extrapolation

The trend: move from **absolute** positional information to **relative** positional information, because what matters for most tasks is the relationship between tokens (how far apart are they?) not their absolute position in a document.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.manual_seed(42)
print("Ready.")

## 1. Sinusoidal Positional Encoding (Vaswani et al. 2017)

The original transformer uses a fixed, non-learned PE based on sin and cos:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

**Why sin/cos?** Two key properties:
1. **Unique representation**: each position gets a unique vector
2. **Relative position can be computed via linear transform**: $PE_{pos+k}$ can be expressed as a linear function of $PE_{pos}$, allowing the model to learn "attend to positions $k$ steps away"

**Why different frequencies?** The frequencies form a geometric progression from $1$ to $1/10000$ across embedding dimensions. Low-frequency dimensions encode coarse position (beginning, middle, end of document); high-frequency dimensions encode fine-grained position (nearby tokens). This is analogous to a binary clock: the least significant bit flips at every step, the most significant bit flips rarely.

**The dot product $PE_{pos}^T PE_{pos'}$ depends only on the offset $(pos - pos')$**, not on the absolute positions. This is a key theoretical property that motivates its use.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Precompute the PE table
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        # Frequencies: 1/10000^(2i/d_model)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) *
            -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # Register as buffer (not a parameter, but moves with model.to(device))
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, T, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# Visualize sinusoidal PE
d_model, max_len = 128, 100
spe = SinusoidalPositionalEncoding(d_model, max_len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap of PE vectors
pe_matrix = spe.pe[0].numpy()  # (max_len, d_model)
im = axes[0].imshow(pe_matrix[:50, :64], cmap='RdBu', aspect='auto')
axes[0].set_xlabel("Embedding dimension")
axes[0].set_ylabel("Position")
axes[0].set_title("Sinusoidal PE: first 50 positions, 64 dims")
plt.colorbar(im, ax=axes[0])

# Show how specific dimensions oscillate
for dim in [0, 4, 8, 16, 32]:
    axes[1].plot(pe_matrix[:50, dim], label=f"dim {dim}")
axes[1].set_xlabel("Position")
axes[1].set_ylabel("PE value")
axes[1].set_title("Different dimensions = different frequencies")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("sinusoidal_pe.png", dpi=120, bbox_inches='tight')
plt.show()

## 2. Learned Positional Embeddings

The simplest approach: treat positions exactly like tokens and learn an embedding for each.

$$PE_\text{learned}(pos) = E_{\text{pos}}[pos]$$

where $E_{\text{pos}} \in \mathbb{R}^{\text{max\_len} \times d_{\text{model}}}$ is a learned embedding matrix.

**Advantages:** Simple to implement; can learn the exact structure useful for the training data; slightly better perplexity than sinusoidal in controlled comparisons.

**Critical limitation:** Position embeddings only exist up to `max_len`. Input sequences longer than this will either crash or produce garbage — there are no position embeddings for those positions. This is the "hard context window" problem. GPT-2, GPT-3, and early BERT all use learned embeddings, which is why they had hard context limits (512-2048 tokens).

**Mitigation:** Some models interpolate position embeddings for longer sequences (e.g., "positional interpolation" for extending LLaMA-1 context).

In [ ]:
class LearnedPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)
        self.max_len = max_len

    def forward(self, x):
        B, T, _ = x.shape
        if T > self.max_len:
            raise ValueError(f"Sequence length {T} exceeds max_len {self.max_len}")
        positions = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)
        return self.dropout(x + self.pos_emb(positions))

# Compare sinusoidal vs learned: cosine similarity between position pairs
d_model = 128
spe = SinusoidalPositionalEncoding(d_model, max_len=200, dropout=0.0)
lpe = LearnedPositionalEmbedding(d_model, max_len=200, dropout=0.0)

sin_pe = spe.pe[0]       # (200, 128)
learned_pe = lpe.pos_emb.weight.data  # (200, 128) random init

# Compute cosine similarity matrix for first 50 positions
def cosine_sim_matrix(emb, n=50):
    emb_n = F.normalize(emb[:n], dim=-1)
    return (emb_n @ emb_n.T).numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, mat, title in zip(axes,
                           [cosine_sim_matrix(sin_pe), cosine_sim_matrix(learned_pe)],
                           ["Sinusoidal PE", "Learned PE (random init)"]):
    im = ax.imshow(mat, cmap='RdBu', vmin=-1, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Position j")
    ax.set_ylabel("Position i")
    plt.colorbar(im, ax=ax)

plt.suptitle("Cosine similarity between position embeddings (pos 0-49)", fontsize=12)
plt.tight_layout()
plt.savefig("pe_similarity.png", dpi=120, bbox_inches='tight')
plt.show()
print("Sinusoidal shows band structure (nearby = similar); learned PE is random at init")

## 3. RoPE: Rotary Position Embeddings

RoPE (Su et al. 2021, "RoFormer") is the current state-of-the-art positional encoding used in LLaMA, Mistral, Gemma, Falcon, Qwen, and most modern LLMs.

**Core idea:** Instead of adding positional information to token embeddings, RoPE rotates the query and key vectors before computing attention. Pairs of dimensions $(q_{2i}, q_{2i+1})$ are rotated by angle $\theta_i \cdot pos$ where $\theta_i = 10000^{-2i/d}$:

$$\mathbf{R}(\theta_i, pos) = \begin{pmatrix} \cos(pos \cdot \theta_i) & -\sin(pos \cdot \theta_i) \\ \sin(pos \cdot \theta_i) & \cos(pos \cdot \theta_i) \end{pmatrix}$$

The full rotation $f_q(\mathbf{q}, pos)$ applies this independently to each pair of dimensions.

**The key property:** The dot product $\langle f_q(\mathbf{q}_m, m), f_k(\mathbf{k}_n, n) \rangle$ depends only on the relative offset $(m - n)$:

$$q_m^T k_n = g(\mathbf{q}, \mathbf{k}, m - n)$$

This means attention automatically encodes relative positions — no additional learned parameters needed.

**Why RoPE enables longer context:** Because it encodes relative rather than absolute position, and the rotation frequencies can be scaled (RoPE scaling / YaRN) to extrapolate to longer sequences. The "context length extension" literature is largely about adjusting RoPE frequencies.

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    """
    RoPE: Rotary Position Embedding.
    Applied to Q and K before attention computation.
    """
    def __init__(self, d_k, base=10000):
        super().__init__()
        assert d_k % 2 == 0
        self.d_k = d_k
        # Precompute inverse frequencies: theta_i = 1 / base^(2i/d_k)
        inv_freq = 1.0 / (base ** (torch.arange(0, d_k, 2).float() / d_k))
        self.register_buffer('inv_freq', inv_freq)

    def get_rotary_matrix(self, seq_len, device):
        """Returns cos and sin matrices: (seq_len, d_k/2)"""
        t = torch.arange(seq_len, device=device).float()
        # (seq_len, d_k/2) = outer product of positions and frequencies
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)  # (seq_len, d_k)
        return emb.cos(), emb.sin()

    @staticmethod
    def rotate_half(x):
        """Rotate pairs: (x1, x2, x3, x4) → (-x3, -x4, x1, x2) for d_k=4"""
        half = x.shape[-1] // 2
        x1 = x[..., :half]
        x2 = x[..., half:]
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, x, offset=0):
        """
        x: (B, n_heads, T, d_k)
        Returns x with rotary position encoding applied.
        """
        B, H, T, d_k = x.shape
        cos, sin = self.get_rotary_matrix(offset + T, x.device)
        cos = cos[offset:offset+T].unsqueeze(0).unsqueeze(0)  # (1,1,T,d_k)
        sin = sin[offset:offset+T].unsqueeze(0).unsqueeze(0)
        return x * cos + self.rotate_half(x) * sin

# Test RoPE
d_k = 64
rope = RotaryPositionalEmbedding(d_k)
B, H, T = 2, 8, 16
Q = torch.randn(B, H, T, d_k)
K = torch.randn(B, H, T, d_k)

Q_rot = rope(Q)
K_rot = rope(K)

# Verify: the dot product Q[i] @ K[j] should depend only on (i-j)
print("RoPE dot-product relative position property:")
for pos_i, pos_j in [(0, 0), (1, 1), (5, 5), (0, 3), (1, 4), (2, 5)]:
    q_i = Q_rot[0, 0, pos_i].unsqueeze(0)  # (1, d_k)
    k_j = K_rot[0, 0, pos_j].unsqueeze(0)
    dot = (q_i * k_j).sum().item()
    offset = pos_i - pos_j
    print(f"  Q[{pos_i}]·K[{pos_j}] (offset={offset:+d}): {dot:.4f}")
print("Pairs with same offset should have similar dot product magnitude (up to content variation)")

## 4. ALiBi: Attention with Linear Biases

ALiBi (Press et al. 2021, "Train Short, Test Long") takes a different approach: instead of modifying Q and K, it adds a fixed linear bias to attention scores based on relative distance:

$$\text{score}_{ij} = \frac{q_i \cdot k_j}{\sqrt{d_k}} - m \cdot |i - j|$$

where $m$ is a head-specific slope. Heads with larger $m$ focus on nearby context; heads with smaller $m$ attend globally.

**The slope schedule:** For $h$ heads, slopes are set to $\{2^{-8/h}, 2^{-8 \cdot 2/h}, \ldots, 2^{-8}\}$.

**Why ALiBi extrapolates well:** The linear bias gives the model an inductive bias that "closer is more relevant." This generalizes naturally to longer sequences — the bias for position $T+100$ is just a slightly larger linear penalty, not an unseen absolute position. BLOOM (176B parameter model) uses ALiBi.

In [ ]:
def get_alibi_slopes(n_heads):
    """Compute ALiBi slopes for each head."""
    def get_slopes_power_of_2(n):
        start = 2 ** (-(2 ** -(math.log2(n) - 3)))
        ratio = start
        return [start * ratio**i for i in range(n)]

    if math.log2(n_heads).is_integer():
        return get_slopes_power_of_2(n_heads)
    else:
        # For non-power-of-2 heads, use closest power of 2 then fill
        closest_pow2 = 2 ** math.floor(math.log2(n_heads))
        slopes = get_slopes_power_of_2(closest_pow2)
        extra = get_slopes_power_of_2(2 * closest_pow2)[0::2][:n_heads - closest_pow2]
        return slopes + extra

def make_alibi_bias(seq_len, n_heads, device='cpu'):
    """
    Returns ALiBi bias: (1, n_heads, seq_len, seq_len)
    bias[h, i, j] = -slope_h * |i - j| (for i >= j in causal setting)
    """
    slopes = torch.tensor(get_alibi_slopes(n_heads), device=device)  # (h,)
    # Relative positions: (T, T) matrix where entry (i,j) = j - i
    positions = torch.arange(seq_len, device=device)
    rel_pos = positions.unsqueeze(0) - positions.unsqueeze(1)  # (T, T)
    # For causal: only apply to past positions (j <= i → rel_pos <= 0)
    alibi = slopes.unsqueeze(-1).unsqueeze(-1) * rel_pos.unsqueeze(0)  # (h, T, T)
    return alibi.unsqueeze(0)  # (1, h, T, T)

# Visualize ALiBi biases for different heads
n_heads, T = 8, 30
alibi = make_alibi_bias(T, n_heads)
slopes = get_alibi_slopes(n_heads)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for h, ax in enumerate(axes.flat):
    bias = alibi[0, h].numpy()
    im = ax.imshow(bias, cmap='RdBu', aspect='auto')
    ax.set_title(f"Head {h+1} (slope={slopes[h]:.4f})")
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("ALiBi Biases: Smaller slope = more global attention", fontsize=12)
plt.tight_layout()
plt.savefig("alibi_biases.png", dpi=100, bbox_inches='tight')
plt.show()

## 5. Comparing Extrapolation Behavior

The key question: if a model is trained with sequence length 128, what happens at inference with length 256?

- **Learned PE:** Hard crash or garbage for positions > max_len
- **Sinusoidal PE:** Degrades gradually; attention patterns are disrupted at new positions
- **ALiBi:** Extrapolates well by design; the linear bias at new distances is simply larger
- **RoPE:** Decent extrapolation; can be extended with frequency scaling (YaRN, LongRoPE)

In [ ]:
# Simulate extrapolation: measure how PE similarity changes at unseen positions
d_model = 128
train_len = 64
test_positions = list(range(0, 200, 10))

# Sinusoidal: what does the cosine similarity look like beyond training length?
spe = SinusoidalPositionalEncoding(d_model, max_len=500, dropout=0.0)
pe = spe.pe[0]  # (500, 128)

# Reference: cosine sim at training length midpoint
ref_pos = train_len // 2
similarities = []
for pos in test_positions:
    sim = F.cosine_similarity(pe[ref_pos].unsqueeze(0), pe[pos].unsqueeze(0)).item()
    similarities.append(sim)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(test_positions, similarities, 'b-o', markersize=4)
axes[0].axvline(x=train_len, color='red', linestyle='--', label=f'Train length={train_len}')
axes[0].axvline(x=ref_pos, color='green', linestyle='--', label=f'Reference pos={ref_pos}')
axes[0].set_xlabel("Position")
axes[0].set_ylabel(f"Cosine sim with position {ref_pos}")
axes[0].set_title("Sinusoidal PE: similarity decays smoothly (some extrapolation)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RoPE: visualize the rotation angles across positions
rope = RotaryPositionalEmbedding(64)
positions = torch.arange(200).float()
inv_freq = 1.0 / (10000 ** (torch.arange(0, 64, 2).float() / 64))
freqs = torch.outer(positions, inv_freq)  # (200, 32)
cos_vals = freqs.cos().numpy()

for dim in [0, 4, 8, 15]:
    axes[1].plot(cos_vals[:, dim], label=f"dim pair {dim}", alpha=0.8)
axes[1].axvline(x=train_len, color='red', linestyle='--', label=f'Train len={train_len}')
axes[1].set_xlabel("Position")
axes[1].set_ylabel("cos(pos * theta_i)")
axes[1].set_title("RoPE: rotation angles continue smoothly beyond train length")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("extrapolation_comparison.png", dpi=120, bbox_inches='tight')
plt.show()

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Why PE needed | Attention is permutation-invariant; without PE, order is invisible |
| Sinusoidal PE | Fixed, no params; multiple frequencies; some extrapolation; original transformer |
| Learned PE | Simple embedding; better fit; hard cutoff at max training length |
| Relative PE | Encode distance $(i-j)$ not absolute position; more generalizable |
| RoPE | Rotate Q,K by position-dependent angle; dot product = relative position function |
| RoPE base | $\theta_i = 10000^{-2i/d_k}$; increasing base extends context (LongRoPE) |
| ALiBi | Additive linear penalty on attention scores; strong extrapolation; used in BLOOM |
| Context extension | RoPE scaling (interpolation/YaRN) extends context without retraining from scratch |
| Modern default | RoPE in LLaMA, Mistral, Gemma, Qwen; ALiBi in BLOOM; learned in GPT-2/3 |